# 06: Visualization Summary

This notebook provides:
- Comprehensive visualizations of all analysis results
- Summary statistics and key findings
- Publication-ready figures

**Inputs:**
- All result files from previous notebooks

**Outputs:**
- Comprehensive summary plots
- Analysis summary report

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import llps_functions as lf
from pathlib import Path

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 10

## 1. List All Available Results

In [ ]:
# Show all saved results
print("📁 Available result files:\n")
available_files = lf.list_saved_results()

## 2. Load Key Results

In [ ]:
# Load summary data
print("\n🔄 Loading analysis results...\n")

try:
    classification_summary = lf.load_analysis_result('classification_summary', format='json')
    print("✅ Loaded classification summary")
except FileNotFoundError:
    classification_summary = None
    print("⚠️  Classification summary not found")

try:
    interaction_summary = lf.load_analysis_result('interaction_summary', format='json')
    print("✅ Loaded interaction summary")
except FileNotFoundError:
    interaction_summary = None
    print("⚠️  Interaction summary not found")

try:
    enrichment_results = lf.load_analysis_result('enrichment_results', format='json')
    print("✅ Loaded enrichment results")
except FileNotFoundError:
    enrichment_results = None
    print("⚠️  Enrichment results not found")

try:
    network_stats = lf.load_analysis_result('network_stats', format='json')
    print("✅ Loaded network statistics")
except FileNotFoundError:
    network_stats = None
    print("⚠️  Network statistics not found")

try:
    functional_stats = lf.load_analysis_result('functional_group_stats', format='json')
    print("✅ Loaded functional group statistics")
except FileNotFoundError:
    functional_stats = None
    print("⚠️  Functional group statistics not found")

## 3. Generate Comprehensive Summary Figure

In [ ]:
# Create multi-panel summary figure
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Panel 1: Dataset Overview
ax1 = fig.add_subplot(gs[0, 0])
if classification_summary:
    class_dist = classification_summary.get('class_distribution', {})
    if class_dist:
        labels = list(class_dist.keys())
        sizes = list(class_dist.values())
        colors = {'High': '#d62728', 'Medium': '#ff7f0e', 'Low': '#2ca02c', 'Unknown': '#7f7f7f'}
        pie_colors = [colors.get(label, '#1f77b4') for label in labels]
        ax1.pie(sizes, labels=labels, autopct='%1.1f%%', colors=pie_colors, startangle=90)
        ax1.set_title('A. pLLPS Classification', fontweight='bold')

# Panel 2: Enrichment Summary
ax2 = fig.add_subplot(gs[0, 1])
if enrichment_results:
    categories = ['High-High', 'High-Other']
    observed = [enrichment_results.get('high_high_observed', 0), 
                enrichment_results.get('high_other_observed', 0)]
    expected = [enrichment_results.get('high_high_expected', 0), 
                enrichment_results.get('high_other_expected', 0)]
    
    x = np.arange(len(categories))
    width = 0.35
    
    ax2.bar(x - width/2, observed, width, label='Observed', alpha=0.8, color='steelblue')
    ax2.bar(x + width/2, expected, width, label='Expected', alpha=0.8, color='coral')
    ax2.set_ylabel('Interaction Count')
    ax2.set_title(f'B. Interaction Enrichment\n(p={enrichment_results.get("p_value", 0):.2e})', fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(categories, rotation=15, ha='right')
    ax2.legend()

# Panel 3: Enrichment Factor
ax3 = fig.add_subplot(gs[0, 2])
if enrichment_results:
    enrichment_factor = enrichment_results.get('enrichment', 1.0)
    ax3.bar(['Enrichment'], [enrichment_factor], color='steelblue', alpha=0.8)
    ax3.axhline(1.0, color='red', linestyle='--', label='No enrichment')
    ax3.set_ylabel('Fold Enrichment')
    ax3.set_title(f'C. Enrichment Factor', fontweight='bold')
    ax3.set_ylim([0, max(2, enrichment_factor * 1.2)])
    ax3.legend()

# Panel 4: Network Statistics
ax4 = fig.add_subplot(gs[1, :])
if network_stats and network_stats.get('network_stats'):
    ns = network_stats['network_stats']
    stats_text = [
        f"Network Nodes: {ns.get('num_nodes', 'N/A')}",
        f"Network Edges: {ns.get('num_edges', 'N/A')}",
        f"Avg Degree: {ns.get('avg_degree', 0):.2f}",
        f"Density: {ns.get('density', 0):.4f}",
        f"Communities: {network_stats.get('num_communities', 'N/A')}"
    ]
    
    # Hub enrichment
    if network_stats.get('hub_enrichment'):
        he = network_stats['hub_enrichment']
        stats_text.extend([
            f"\nHub Enrichment: {he.get('enrichment', 1.0):.2f}x",
            f"Hub p-value: {he.get('p_value', 1.0):.2e}"
        ])
    
    ax4.text(0.1, 0.5, '\n'.join(stats_text), 
             fontsize=12, verticalalignment='center',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax4.axis('off')
    ax4.set_title('D. Network Statistics', fontweight='bold', loc='left')

# Panel 5: Functional Categories (if available)
ax5 = fig.add_subplot(gs[2, :])
if functional_stats:
    categories = list(functional_stats.keys())[:8]  # Top 8
    mean_pllps = [functional_stats[cat]['mean_pllps'] for cat in categories]
    high_pllps_pct = [functional_stats[cat]['high_pllps_percentage'] for cat in categories]
    
    x = np.arange(len(categories))
    width = 0.35
    
    ax5_twin = ax5.twinx()
    
    bars1 = ax5.bar(x - width/2, mean_pllps, width, label='Mean pLLPS', alpha=0.8, color='steelblue')
    bars2 = ax5_twin.bar(x + width/2, high_pllps_pct, width, label='% High pLLPS', alpha=0.8, color='coral')
    
    ax5.set_xlabel('Functional Category')
    ax5.set_ylabel('Mean pLLPS Score', color='steelblue')
    ax5_twin.set_ylabel('% High pLLPS', color='coral')
    ax5.set_title('E. Functional Group Analysis', fontweight='bold', loc='left')
    ax5.set_xticks(x)
    ax5.set_xticklabels(categories, rotation=45, ha='right')
    ax5.tick_params(axis='y', labelcolor='steelblue')
    ax5_twin.tick_params(axis='y', labelcolor='coral')
    
    # Add legends
    ax5.legend(loc='upper left')
    ax5_twin.legend(loc='upper right')

plt.suptitle('LLPS Protein Interaction Analysis Summary', fontsize=16, fontweight='bold', y=0.995)
plt.savefig('results/comprehensive_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Saved comprehensive summary: results/comprehensive_summary.png")

## 4. Generate Analysis Report

In [ ]:
# Create text summary report
report_lines = []
report_lines.append("="*70)
report_lines.append("LLPS PROTEIN INTERACTION ANALYSIS SUMMARY")
report_lines.append("="*70)
report_lines.append("")

# Dataset Overview
if classification_summary:
    report_lines.append("1. DATASET OVERVIEW")
    report_lines.append("-" * 70)
    report_lines.append(f"   Total proteins: {classification_summary.get('total_proteins', 'N/A')}")
    report_lines.append(f"   High pLLPS proteins: {classification_summary.get('high_pllps_count', 'N/A')} ({classification_summary.get('high_pllps_percentage', 0):.1f}%)")
    report_lines.append(f"   Membrane proteins: {classification_summary.get('membrane_protein_count', 'N/A')} ({classification_summary.get('membrane_protein_percentage', 0):.1f}%)")
    report_lines.append("")

# Interaction Analysis
if interaction_summary:
    report_lines.append("2. STRING INTERACTION ANALYSIS")
    report_lines.append("-" * 70)
    report_lines.append(f"   Proteins queried: {interaction_summary.get('proteins_queried', 'N/A')}")
    report_lines.append(f"   Total interactions found: {interaction_summary.get('total_interactions', 'N/A')}")
    report_lines.append(f"   Complete pairs (both in dataset): {interaction_summary.get('complete_pairs', 'N/A')}")
    report_lines.append(f"   Score threshold: {interaction_summary.get('score_threshold', 'N/A')}")
    report_lines.append("")

# Enrichment Analysis
if enrichment_results:
    report_lines.append("3. ENRICHMENT ANALYSIS")
    report_lines.append("-" * 70)
    report_lines.append(f"   High-High interactions (observed): {enrichment_results.get('high_high_observed', 'N/A')}")
    report_lines.append(f"   High-High interactions (expected): {enrichment_results.get('high_high_expected', 0):.1f}")
    report_lines.append(f"   Enrichment factor: {enrichment_results.get('enrichment', 1.0):.2f}x")
    report_lines.append(f"   Chi-squared p-value: {enrichment_results.get('p_value', 1.0):.4e}")
    
    if enrichment_results.get('p_value', 1.0) < 0.05:
        report_lines.append(f"   ✅ SIGNIFICANT enrichment detected (p < 0.05)")
    else:
        report_lines.append(f"   ⚠️  No significant enrichment (p >= 0.05)")
    report_lines.append("")

# Network Analysis
if network_stats:
    report_lines.append("4. NETWORK ANALYSIS")
    report_lines.append("-" * 70)
    ns = network_stats.get('network_stats', {})
    report_lines.append(f"   Network nodes: {ns.get('num_nodes', 'N/A')}")
    report_lines.append(f"   Network edges: {ns.get('num_edges', 'N/A')}")
    report_lines.append(f"   Average degree: {ns.get('avg_degree', 0):.2f}")
    report_lines.append(f"   Network density: {ns.get('density', 0):.4f}")
    report_lines.append(f"   Communities detected: {network_stats.get('num_communities', 'N/A')}")
    
    if network_stats.get('hub_enrichment'):
        he = network_stats['hub_enrichment']
        report_lines.append(f"")
        report_lines.append(f"   Hub Enrichment:")
        report_lines.append(f"      Enrichment factor: {he.get('enrichment', 1.0):.2f}x")
        report_lines.append(f"      P-value: {he.get('p_value', 1.0):.4e}")
        if he.get('p_value', 1.0) < 0.05:
            report_lines.append(f"      ✅ High pLLPS proteins ARE enriched in hubs")
        else:
            report_lines.append(f"      ⚠️  High pLLPS proteins NOT enriched in hubs")
    report_lines.append("")

# Functional Groups
if functional_stats:
    report_lines.append("5. FUNCTIONAL GROUP ANALYSIS")
    report_lines.append("-" * 70)
    report_lines.append(f"   Number of functional categories: {len(functional_stats)}")
    report_lines.append(f"")
    report_lines.append(f"   Top categories by mean pLLPS:")
    
    sorted_cats = sorted(functional_stats.items(), 
                        key=lambda x: x[1]['mean_pllps'], 
                        reverse=True)[:5]
    
    for cat, stats in sorted_cats:
        report_lines.append(f"      {cat}: mean={stats['mean_pllps']:.3f}, high%={stats['high_pllps_percentage']:.1f}%")
    report_lines.append("")

report_lines.append("="*70)
report_lines.append("Analysis completed successfully!")
report_lines.append("="*70)

# Print report
report_text = "\n".join(report_lines)
print(report_text)

# Save report
with open('results/analysis_summary_report.txt', 'w') as f:
    f.write(report_text)

print("\n✅ Saved report: results/analysis_summary_report.txt")

## 5. List All Generated Plots

In [ ]:
# List all plot files
results_path = Path('results')
plot_files = list(results_path.glob('*.png'))

if plot_files:
    print(f"\n📊 Generated {len(plot_files)} visualization plots:\n")
    for plot_file in sorted(plot_files):
        size = plot_file.stat().st_size
        size_str = f"{size/1024:.1f} KB" if size < 1024*1024 else f"{size/(1024*1024):.1f} MB"
        print(f"   ✅ {plot_file.name} ({size_str})")
else:
    print("\n⚠️  No plot files found")

## Summary

✅ **Completed:**
1. Loaded all analysis results
2. Generated comprehensive summary figure
3. Created text summary report
4. Listed all generated visualizations

**All analysis notebooks completed!**

Results are saved in the `results/` directory and can be loaded for further analysis or publication.